# Logistic Regression Exploration: Tract-Level Arrest Activity

## Purpose

This notebook interprets the baseline logistic regression model for classifying census tracts into elevated and non-elevated arrest activity groups. The model was created by `ml/scripts/train_logistic_regression.py` and is intended to support tract-level analytical review and decision-support interpretation.

This is not crime prediction and it is not individual-level risk scoring. The model works only at the census tract level and should be interpreted as an exploratory analysis of tract-level arrest activity patterns.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "ml").exists():
    repo_root = Path.cwd().parents[1]

dataset_path = repo_root / "ml" / "outputs" / "ml_tract_dataset.csv"
metrics_path = repo_root / "ml" / "outputs" / "logistic_regression_metrics.json"
coefficients_path = repo_root / "ml" / "outputs" / "logistic_regression_coefficients.csv"
predictions_path = repo_root / "ml" / "outputs" / "logistic_regression_predictions.csv"

ml_dataset = pd.read_csv(dataset_path)
with metrics_path.open("r", encoding="utf-8") as file:
    metrics = json.load(file)
coefficients = pd.read_csv(coefficients_path)
predictions = pd.read_csv(predictions_path)

## Dataset Overview

In [ ]:
ml_dataset.shape

In [ ]:
target_counts = ml_dataset["elevated_arrest_activity_flag"].value_counts().sort_index()
target_counts

In [ ]:
target_share = ml_dataset["elevated_arrest_activity_flag"].value_counts(normalize=True).sort_index()
target_share.rename(index={0: "non_elevated_share", 1: "elevated_share"})

## Target Definition

- `elevated_arrest_activity_flag = 1` means the tract is in the top 25 percent by arrests per 1,000 residents.
- `elevated_arrest_activity_flag = 0` means all other tracts.
- `arrests_per_1000_population` was excluded from model features to avoid direct target leakage.

## Feature Review

In [ ]:
pd.DataFrame({"feature_columns": metrics["feature_columns"]})

In [ ]:
pd.DataFrame({"excluded_columns": metrics["excluded_columns"]})

Raw arrest-volume variables were excluded because they are too directly tied to the target definition or overall arrest volume. This helps reduce leakage and keeps the model focused on tract-level contextual and composition features rather than direct rate or count proxies.

Excluded raw activity and direct-rate variables include:

- `total_arrests`
- `arrests_per_1000_population`
- `arrests_density_per_sq_mi`
- `arrest_activity_share`
- `arrest_weekend_events`
- `arrest_evening_night_events`
- `arrest_night_events`
- `felony_arrests`
- `misdemeanor_arrests`

## Model Metrics

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
pd.Series({name: metrics.get(name) for name in metric_names}, name="value")

In [ ]:
confusion = np.array(metrics["confusion_matrix"])
pd.DataFrame(
    confusion,
    index=["actual_non_elevated", "actual_elevated"],
    columns=["predicted_non_elevated", "predicted_elevated"],
)

## Confusion Matrix Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(confusion)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0, 1], labels=["Non-elevated", "Elevated"])
ax.set_yticks([0, 1], labels=["Non-elevated", "Elevated"])

for row in range(confusion.shape[0]):
    for col in range(confusion.shape[1]):
        ax.text(col, row, confusion[row, col], ha="center", va="center")

fig.colorbar(image, ax=ax)
plt.tight_layout()

## Coefficient Interpretation

The coefficients below are sorted by absolute coefficient size. Because the model pipeline scales numeric features before fitting the logistic regression, these coefficients reflect associations after standardization. They should not be interpreted causally. A positive coefficient is associated with higher model probability for elevated arrest activity, while a negative coefficient is associated with lower model probability, holding the model structure constant.

In [ ]:
top_coefficients = coefficients.sort_values("absolute_coefficient", ascending=False).head(15)
top_coefficients

## Coefficient Plot

In [ ]:
plot_data = top_coefficients.sort_values("absolute_coefficient")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot_data["feature"], plot_data["coefficient"])
ax.set_title("Top 15 Logistic Regression Coefficients")
ax.set_xlabel("Coefficient")
ax.set_ylabel("Feature")
plt.tight_layout()

## Prediction Review

In [ ]:
display_columns = [
    column
    for column in ["tract_geoid", "actual", "predicted", "predicted_probability_elevated"]
    if column in predictions.columns
]
predictions.sort_values("predicted_probability_elevated", ascending=False)[display_columns]

## Limitations

- The dataset is small, with 68 census tracts.
- This model is exploratory and should be treated as a baseline interpretation tool.
- The high ROC AUC should be interpreted cautiously because the sample is limited and the train/test split is small.
- Demographic variables require responsible interpretation and should not be used to stigmatize communities.
- Arrest data reflects enforcement and administrative activity, not direct harm.
- Model outputs should support analytical review, not operational directives.

## Portfolio Talking Points

- I created a reproducible ML pipeline separate from the dashboard.
- I avoided direct leakage by excluding arrest-rate and raw arrest-count variables.
- I used logistic regression first because the target is binary and explainable.
- I exported metrics, coefficients, predictions, and a model artifact.
- I used the notebook for interpretation while keeping the training script as the reproducible artifact.